In [0]:
-- step 1 Distinct patients 
-- Using step 1, distinct service dates in the specified time period (5 year dx)
-- Group by on patient id, 
-- Use diagnosis code column to get patients having the diagnosis codes 2>= for black codes.
-- rest of them will be non severe patients.


In [0]:
select distinct a.patient_id, count(distinct b.SERVICE_DATE) as fill_date
from com_edp_prd.cmpa_insights_internal_schema.patient360 a
left join com_edp_prd.com_raw.kom_medical_events b
on a.patient_id = b.patient_id and b.SERVICE_DATE between '2020-08-01' and '2025-07-31'
group by a.PATIENT_ID

In [0]:

WITH base AS (
  SELECT DISTINCT a.PATIENT_ID,
         b.SERVICE_DATE,
         b.DIAGNOSIS_CODES
  FROM com_edp_prd.cmpa_insights_internal_schema.patient360 a
  LEFT JOIN com_edp_prd.com_raw.kom_medical_events b
    ON a.PATIENT_ID = b.PATIENT_ID
   AND b.SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),

flagged AS (
  SELECT
    PATIENT_ID,
    SERVICE_DATE,
    CASE
      WHEN DIAGNOSIS_CODES ILIKE '%G910%'
        OR DIAGNOSIS_CODES ILIKE '%G911%'
        OR DIAGNOSIS_CODES ILIKE '%G912%'
        OR DIAGNOSIS_CODES ILIKE '%G913%'
        OR DIAGNOSIS_CODES ILIKE '%G914%'
        OR DIAGNOSIS_CODES ILIKE '%G918%'
        OR DIAGNOSIS_CODES ILIKE '%G919%'
        OR DIAGNOSIS_CODES ILIKE '%Q038%'
        OR DIAGNOSIS_CODES ILIKE '%Q039%'
        OR DIAGNOSIS_CODES ILIKE '%Q050%'
        OR DIAGNOSIS_CODES ILIKE '%Q051%'
        OR DIAGNOSIS_CODES ILIKE '%Q052%'
        OR DIAGNOSIS_CODES ILIKE '%Q053%'
        OR DIAGNOSIS_CODES ILIKE '%Q054%'
        OR DIAGNOSIS_CODES ILIKE '%Q055%'
        OR DIAGNOSIS_CODES ILIKE '%Q056%'
        OR DIAGNOSIS_CODES ILIKE '%Q057%'
        OR DIAGNOSIS_CODES ILIKE '%Q058%'
        OR DIAGNOSIS_CODES ILIKE '%Q0700%'
        OR DIAGNOSIS_CODES ILIKE '%Q0702%'
        OR DIAGNOSIS_CODES ILIKE '%Q0703%'
        OR DIAGNOSIS_CODES ILIKE '%F445%'
        OR DIAGNOSIS_CODES ILIKE '%F639%'
        OR DIAGNOSIS_CODES ILIKE '%F70%'
        OR DIAGNOSIS_CODES ILIKE '%F71%'
        OR DIAGNOSIS_CODES ILIKE '%F72%'
        OR DIAGNOSIS_CODES ILIKE '%F73%'
        OR DIAGNOSIS_CODES ILIKE '%F78%'
        OR DIAGNOSIS_CODES ILIKE '%F78A1%'
        OR DIAGNOSIS_CODES ILIKE '%F78A9%'
        OR DIAGNOSIS_CODES ILIKE '%F79%'
        OR DIAGNOSIS_CODES ILIKE '%F800%'
        OR DIAGNOSIS_CODES ILIKE '%F801%'
        OR DIAGNOSIS_CODES ILIKE '%F802%'
        OR DIAGNOSIS_CODES ILIKE '%F804%'
        OR DIAGNOSIS_CODES ILIKE '%F8081%'
        OR DIAGNOSIS_CODES ILIKE '%F8082%'
        OR DIAGNOSIS_CODES ILIKE '%F8089%'
        OR DIAGNOSIS_CODES ILIKE '%F809%'
        OR DIAGNOSIS_CODES ILIKE '%F810%'
        OR DIAGNOSIS_CODES ILIKE '%F812%'
        OR DIAGNOSIS_CODES ILIKE '%F8181%'
        OR DIAGNOSIS_CODES ILIKE '%F8189%'
        OR DIAGNOSIS_CODES ILIKE '%F819%'
        OR DIAGNOSIS_CODES ILIKE '%F82%'
        OR DIAGNOSIS_CODES ILIKE '%F840%'
        OR DIAGNOSIS_CODES ILIKE '%F843%'
        OR DIAGNOSIS_CODES ILIKE '%F845%'
        OR DIAGNOSIS_CODES ILIKE '%F848%'
        OR DIAGNOSIS_CODES ILIKE '%F849%'
        OR DIAGNOSIS_CODES ILIKE '%F88%'
        OR DIAGNOSIS_CODES ILIKE '%F89%'
        OR DIAGNOSIS_CODES ILIKE '%R6250%'
        OR DIAGNOSIS_CODES ILIKE '%R620%'
        OR DIAGNOSIS_CODES ILIKE '%R6251%'
        OR DIAGNOSIS_CODES ILIKE '%R6259%'
        OR DIAGNOSIS_CODES ILIKE '%R62%'
      THEN 1 ELSE 0
    END AS has_black_code
  FROM base
)

SELECT
  PATIENT_ID,
  COUNT(DISTINCT SERVICE_DATE) AS fill_date,
  SUM(has_black_code) AS black_dx_row_count,
  COUNT(DISTINCT CASE WHEN has_black_code = 1 THEN SERVICE_DATE END) AS black_dx_distinct_dates,
  CASE WHEN COUNT(DISTINCT CASE WHEN has_black_code = 1 THEN SERVICE_DATE END) >= 2 THEN 'Severe' ELSE 'Non-severe' END AS severity
FROM flagged
GROUP BY PATIENT_ID
ORDER BY black_dx_distinct_dates DESC, PATIENT_ID;


In [0]:

WITH base AS (
  SELECT DISTINCT a.PATIENT_ID,
         b.SERVICE_DATE,
         b.DIAGNOSIS_CODES
  FROM com_edp_prd.cmpa_insights_internal_schema.patient360 a
  LEFT JOIN com_edp_prd.com_raw.kom_medical_events b
    ON a.PATIENT_ID = b.PATIENT_ID
   AND b.SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),

flagged AS (
  SELECT
    PATIENT_ID,
    SERVICE_DATE,
    CASE
      WHEN DIAGNOSIS_CODES IS NOT NULL AND (
           DIAGNOSIS_CODES ILIKE '%|G910|%' OR DIAGNOSIS_CODES ILIKE '%|G911|%' OR DIAGNOSIS_CODES ILIKE '%|G912|%'
        OR DIAGNOSIS_CODES ILIKE '%|G913|%' OR DIAGNOSIS_CODES ILIKE '%|G914|%' OR DIAGNOSIS_CODES ILIKE '%|G918|%'
        OR DIAGNOSIS_CODES ILIKE '%|G919|%' OR DIAGNOSIS_CODES ILIKE '%|Q038|%' OR DIAGNOSIS_CODES ILIKE '%|Q039|%'
        OR DIAGNOSIS_CODES ILIKE '%|Q050|%' OR DIAGNOSIS_CODES ILIKE '%|Q051|%' OR DIAGNOSIS_CODES ILIKE '%|Q052|%'
        OR DIAGNOSIS_CODES ILIKE '%|Q053|%' OR DIAGNOSIS_CODES ILIKE '%|Q054|%' OR DIAGNOSIS_CODES ILIKE '%|Q055|%'
        OR DIAGNOSIS_CODES ILIKE '%|Q056|%' OR DIAGNOSIS_CODES ILIKE '%|Q057|%' OR DIAGNOSIS_CODES ILIKE '%|Q058|%'
        OR DIAGNOSIS_CODES ILIKE '%|Q0700|%' OR DIAGNOSIS_CODES ILIKE '%|Q0702|%' OR DIAGNOSIS_CODES ILIKE '%|Q0703|%'
        OR DIAGNOSIS_CODES ILIKE '%|F445|%' OR DIAGNOSIS_CODES ILIKE '%|F639|%' OR DIAGNOSIS_CODES ILIKE '%|F70|%'
        OR DIAGNOSIS_CODES ILIKE '%|F71|%' OR DIAGNOSIS_CODES ILIKE '%|F72|%' OR DIAGNOSIS_CODES ILIKE '%|F73|%'
        OR DIAGNOSIS_CODES ILIKE '%|F78|%' OR DIAGNOSIS_CODES ILIKE '%|F78A1|%' OR DIAGNOSIS_CODES ILIKE '%|F78A9|%'
        OR DIAGNOSIS_CODES ILIKE '%|F79|%' OR DIAGNOSIS_CODES ILIKE '%|F800|%' OR DIAGNOSIS_CODES ILIKE '%|F801|%'
        OR DIAGNOSIS_CODES ILIKE '%|F802|%' OR DIAGNOSIS_CODES ILIKE '%|F804|%' OR DIAGNOSIS_CODES ILIKE '%|F8081|%'
        OR DIAGNOSIS_CODES ILIKE '%|F8082|%' OR DIAGNOSIS_CODES ILIKE '%|F8089|%' OR DIAGNOSIS_CODES ILIKE '%|F809|%'
        OR DIAGNOSIS_CODES ILIKE '%|F810|%' OR DIAGNOSIS_CODES ILIKE '%|F812|%' OR DIAGNOSIS_CODES ILIKE '%|F8181|%'
        OR DIAGNOSIS_CODES ILIKE '%|F8189|%' OR DIAGNOSIS_CODES ILIKE '%|F819|%' OR DIAGNOSIS_CODES ILIKE '%|F82|%'
        OR DIAGNOSIS_CODES ILIKE '%|F840|%' OR DIAGNOSIS_CODES ILIKE '%|F843|%' OR DIAGNOSIS_CODES ILIKE '%|F845|%'
        OR DIAGNOSIS_CODES ILIKE '%|F848|%' OR DIAGNOSIS_CODES ILIKE '%|F849|%' OR DIAGNOSIS_CODES ILIKE '%|F88|%'
        OR DIAGNOSIS_CODES ILIKE '%|F89|%' OR DIAGNOSIS_CODES ILIKE '%|R6250|%' OR DIAGNOSIS_CODES ILIKE '%|R620|%'
        OR DIAGNOSIS_CODES ILIKE '%|R6251|%' OR DIAGNOSIS_CODES ILIKE '%|R6259|%' OR DIAGNOSIS_CODES ILIKE '%|R62|%'
      )
      THEN 1 ELSE 0
    END AS has_black_code
  FROM base
)

SELECT
  PATIENT_ID,
  COUNT(DISTINCT SERVICE_DATE) AS count_fill_date,  
  COUNT(DISTINCT CASE WHEN has_black_code = 1 THEN SERVICE_DATE END) AS black_dx_distinct_dates,
  CASE WHEN COUNT(DISTINCT CASE WHEN has_black_code = 1 THEN SERVICE_DATE END) >= 2
       THEN 'Severe' ELSE 'Attenuated' END AS severity
FROM flagged
GROUP BY PATIENT_ID
ORDER BY black_dx_distinct_dates DESC, PATIENT_ID;


In [0]:
select *
from com_edp_prd.com_raw.kom_medical_events
where PATIENT_ID in ('TWL2BW0Z', 'J0CRCXC9', '9GQLZ50J')
and SERVICE_DATE between '2020-08-01' and '2025-07-31'
sort by PATIENT_ID

In [0]:
select * from com_edp_prd.com_raw.vod_hcp limit 3